# Graphs and workflows

Read the [guide](README.md). Install the notebooks, multiagent and partner groups for the full tier. All model calls below replay offline; local A2A transports still use real child processes.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_graph_basics

Predict the result, then run the [script](examples/01_graph_basics.py).


In [ ]:
"""Graph nodes and edges: Nimbus Tier 4 lesson.

Run: uv run python modules/17_graphs/examples/01_graph_basics.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

from common.coordination import report
from common.graph_lessons import basics
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    # KEY IDEA: the application declares the dependency; the model writes within a node.
    graph = basics(cfg)
    result = graph("Research then draft a Porto trip")
    report(result, cfg)
    for name, node in result.results.items():
        print(name, str(node.result))
    complete()


run_lesson(main, "modules/17_graphs/examples/01_graph_basics.py")

## 02_parallel_search

Predict the result, then run the [script](examples/02_parallel_search.py).


In [ ]:
"""Parallel branches with fan-in: Nimbus Tier 4 lesson.

Run: uv run python modules/17_graphs/examples/02_parallel_search.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

from common.coordination import report
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.trip_graph import group_trip_graph


def main() -> None:
    _, cfg = lesson_settings()
    graph, state, request = group_trip_graph(cfg, budget_cents=1, search_delay=0.05)
    result = graph(request.model_dump_json())
    flight, hotel = state["intervals"]["flight"], state["intervals"]["hotel"]
    # KEY IDEA: overlap proves concurrency; the budget waits for both branches.
    print("Search intervals overlap:", max(flight[0], hotel[0]) < min(flight[1], hotel[1]))
    print("Joined quote:", state["review"].quote.model_dump())
    report(result, cfg)
    complete()


run_lesson(main, "modules/17_graphs/examples/02_parallel_search.py")

## 03_conditional_budget

Predict the result, then run the [script](examples/03_conditional_budget.py).


In [ ]:
"""Conditional routes and bounded loops: Nimbus Tier 4 lesson.

Run: uv run python modules/17_graphs/examples/03_conditional_budget.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

from common.coordination import report
from common.graph_lessons import bounded_loop
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.trip_graph import run_group_trip


def main() -> None:
    _, cfg = lesson_settings()
    result, state = run_group_trip(cfg, budget_cents=1)
    print("Within budget:", state["within_budget"])
    report(result, cfg)
    # KEY IDEA: a retry edge needs an execution ceiling and a deadline.
    graph, attempts = bounded_loop(maximum=3)
    result = graph("Revise an impossible budget")
    print("Revision attempts:", attempts["attempts"])
    report(result, cfg)
    complete()


run_lesson(main, "modules/17_graphs/examples/03_conditional_budget.py")

## 04_nested_patterns

Predict the result, then run the [script](examples/04_nested_patterns.py).


In [ ]:
"""Nested graph and swarm: Nimbus Tier 4 lesson.

Run: uv run python modules/17_graphs/examples/04_nested_patterns.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

from common.coordination import report
from common.graph_lessons import basics
from common.lessons import complete, lesson_settings
from common.pretty import run_cli


def main() -> None:
    _, cfg = lesson_settings()
    for pattern in ("graph", "swarm"):
        result = basics(cfg, nested=pattern)("Make a Porto draft")
        print("Nested pattern:", pattern)
        # KEY IDEA: outer limits and inner limits are separate; do not sum usage twice.
        report(result, cfg)
    complete()


run_lesson(main, "modules/17_graphs/examples/04_nested_patterns.py")

## 05_workflow_comparison

Predict the result, then run the [script](examples/05_workflow_comparison.py).


In [ ]:
"""Workflow tool metadata versus Graph execution: Nimbus Tier 4 lesson.

Run: uv run python modules/17_graphs/examples/05_workflow_comparison.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import json

from common.coordination import report
from common.graph_lessons import basics
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.workflow_inspection import inspect_workflow


def main() -> None:
    _, cfg = lesson_settings()
    print(json.dumps(inspect_workflow(), indent=2))
    # KEY IDEA: Graph executes in this invocation; workflow owns persistent background tasks.
    report(basics(cfg)("Research and draft Porto"), cfg)
    complete()


run_lesson(main, "modules/17_graphs/examples/05_workflow_comparison.py")

## 06_approval_pipeline

Predict the result, then run the [script](examples/06_approval_pipeline.py).


In [ ]:
"""Human review before confirmation draft: Nimbus Tier 4 lesson.

Run: uv run python modules/17_graphs/examples/06_approval_pipeline.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import argparse

from common.coordination import report
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.trip_graph import run_group_trip


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--decision", choices=["approve", "reject", "ask"], default="reject")
    parser.add_argument("--budget-cents", type=int, default=100000)
    args, cfg = lesson_settings(parser)
    # KEY IDEA: the host records approval of the exact quote; model text grants no authority.
    result, state = run_group_trip(cfg, decision=args.decision, budget_cents=args.budget_cents)
    report(result, cfg)
    print("Draft:", state["draft"] or "None: review not approved or over budget")
    complete()


run_lesson(main, "modules/17_graphs/examples/06_approval_pipeline.py")

## group_trip_pipeline

Predict the result, then run the [script](use_case/group_trip_pipeline.py).


In [ ]:
"""Group trip pipeline: Nimbus Tier 4 lesson.

Run: uv run python modules/17_graphs/use_case/group_trip_pipeline.py --mock
Expected: Lesson complete. Mock fixtures cost $0; Ollama has no API fee.
Live inference uses configured rates and bounded calls. AWS resources created: none.
"""

import argparse

from common.coordination import report
from common.lessons import complete, lesson_settings
from common.pretty import run_cli
from common.trip_graph import run_group_trip


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--decision", choices=["approve", "reject", "ask"], default="reject")
    parser.add_argument("--budget-cents", type=int, default=100000)
    args, cfg = lesson_settings(parser)
    # KEY IDEA: the host records approval of the exact quote; model text grants no authority.
    result, state = run_group_trip(cfg, decision=args.decision, budget_cents=args.budget_cents)
    report(result, cfg)
    print("Draft:", state["draft"] or "None: review not approved or over budget")
    complete()


run_lesson(main, "modules/17_graphs/use_case/group_trip_pipeline.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md). Module 17 defaults to rejection; use the standalone --decision ask command to review and approve a draft.
